## 🎯 Learning Objectives
* Design and implement a multi-agent system for automated code review and debugging.
* Utilize AutoGen's `GroupChat` and `GroupChatManager` to orchestrate complex agent workflows.
* Configure specialized `AssistantAgent` roles for distinct tasks like code review, debugging, and verification.
* Implement dynamic termination conditions based on agent output within a group chat.
* Understand how to pass context and manage state across multiple agents in a collaborative setting.


## Exercise: Build a Multi-Agent Debugging and Code Review System

**Lesson ID:** ADV02-L09

### Task Description

In this exercise, you will design and implement a sophisticated multi-agent system using AutoGen to automate the process of code review, debugging, and verification. Imagine a scenario where a developer submits a piece of code, and a team of AI agents collaboratively reviews it, identifies issues, proposes fixes, and then verifies those fixes.

Your goal is to create a system that can take a problematic Python code snippet, have an AI agent review it, another AI agent attempt to fix it based on the review, and a final AI agent verify the correctness of the fix. The system should iterate until a satisfactory fix is found or a maximum number of attempts is reached.

### Requirements

1.  **Developer Agent (`UserProxyAgent`):**
    *   Initiates the process by submitting a Python code snippet that contains a known bug or potential improvement.
    *   `human_input_mode` should be set to `NEVER` for fully automated execution.

2.  **Code Reviewer Agent (`AssistantAgent`):**
    *   Receives the initial code from the Developer Agent.
    *   Analyzes the code for bugs, style violations, potential performance issues, or security vulnerabilities.
    *   Provides detailed feedback to the group, highlighting specific issues and suggesting improvements. If no issues are found, it should clearly state that the "Code looks good."

3.  **Debugger/Fixer Agent (`AssistantAgent`):**
    *   Monitors the group chat for feedback from the Code Reviewer.
    *   If issues are reported, this agent takes the original code and the reviewer's feedback to generate a corrected version of the code.
    *   Presents the corrected code to the group for further review.

4.  **Verifier Agent (`AssistantAgent`):**
    *   Monitors the group chat for corrected code from the Debugger/Fixer Agent.
    *   "Tests" the corrected code. For this exercise, "testing" can be an LLM-based reasoning process where the agent evaluates if the proposed fix addresses the original issues and doesn't introduce new ones. It should not actually execute the code in a sandbox for this exercise, but rather reason about its correctness.
    *   Reports "PASS" if the fix appears correct and robust, or "FAIL" with reasons if it doesn't.

5.  **Orchestration (`GroupChat` and `GroupChatManager`):**
    *   Use `GroupChat` to define the collaborative environment for these agents.
    *   Use `GroupChatManager` to manage the flow of conversation and agent turns.

6.  **Termination Condition:**
    *   The `GroupChatManager` should terminate the conversation when the Verifier Agent reports "PASS" (case-insensitive) in its message.
    *   Implement a `max_round` for the `GroupChat` to prevent infinite loops (e.g., 10 rounds).

7.  **LLM Configuration:**
    *   Ensure your `config_list` is correctly set up to use your preferred LLM provider (e.g., OpenAI, Azure OpenAI, local models).
    *   Consider if different agents might benefit from slightly different LLM parameters (e.g., `temperature`).

### Evaluation Criteria

*   **Correctness:** Does the system correctly identify and attempt to fix a simple bug?
*   **Agent Interaction:** Are the agents interacting logically and passing context effectively?
*   **AutoGen Usage:** Is `GroupChat` and `GroupChatManager` used appropriately for orchestration?
*   **Termination:** Does the system terminate correctly upon successful verification or reaching `max_round`?
*   **Code Quality:** Is the code clean, well-commented, and easy to understand?
*   **Robustness:** Can it handle a basic bug-fixing scenario and provide reasonable output?


In [ ]:
import autogen
import os

# --- Configuration for LLMs ---
# Ensure you have your OAI_CONFIG_LIST environment variable set up
# or a local OAI_CONFIG_LIST.json file.
# Example OAI_CONFIG_LIST.json structure:
# [
#     {
#         "model": "gpt-4o-2024-05-13",
#         "api_key": "YOUR_OPENAI_API_KEY"
#     },
#     {
#         "model": "gpt-3.5-turbo",
#         "api_key": "YOUR_OPENAI_API_KEY"
#     }
# ]

# For 2026, we assume advanced models like 'gpt-4o' or similar are standard.
# We'll use a single config list for simplicity, but in practice, different
# agents might use different models or parameters.

config_list = autogen.config_list_from_json(
    "OAI_CONFIG_LIST",
    filter_dict={
        "model": ["gpt-4o", "gpt-4o-2024-05-13", "gpt-3.5-turbo", "gpt-4-turbo-preview"]
    }
)

llm_config = {
    "timeout": 600,
    "cache_seed": 42, # For reproducibility
    "config_list": config_list,
    "temperature": 0.1 # Lower temperature for more deterministic, factual responses
}

# --- Sample Problematic Code --- 
# This is the code snippet the Developer Agent will submit.
# It contains a simple bug: integer division in Python 2 style, 
# and a potential off-by-one error for average calculation if list is empty.
problematic_code = """
def calculate_average(numbers):
    # This function is supposed to calculate the average of a list of numbers.
    # It has a bug related to integer division and might not handle empty lists gracefully.
    if len(numbers) == 0:
        return 0 # Should probably raise an error or return None, but 0 is a common default.
    total = sum(numbers)
    count = len(numbers)
    average = total / count # Potential integer division issue in some Python versions/contexts
    return average

# Example usage:
my_list = [1, 2, 3, 4, 5]
print(f"The average is: {calculate_average(my_list)}")

my_empty_list = []
print(f"The average of an empty list is: {calculate_average(my_empty_list)}")
"""

print("Setup complete. LLM configuration loaded and problematic code defined.")


### Your Implementation

Now it's your turn! In the cell below, implement the multi-agent system as described in the requirements. Create the Developer, Code Reviewer, Debugger/Fixer, and Verifier agents. Orchestrate their interaction using `GroupChat` and `GroupChatManager` to review, fix, and verify the `problematic_code` provided in the setup cell.

Remember to include comments explaining your design choices and how each agent contributes to the overall goal. Pay close attention to the termination conditions and how agents pass information.


In [ ]:
# --- Agent Definitions ---

# 1. Developer Agent (UserProxyAgent)
# This agent initiates the conversation by submitting the code.
# human_input_mode='NEVER' ensures full automation.
# code_execution_config is set to False as it's not meant to execute code directly in this role.
developer = autogen.UserProxyAgent(
    name="Developer",
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config=False, # Developer just submits, doesn't execute in this flow
    llm_config=llm_config,
    system_message="You are a software developer who writes Python code and submits it for review. You will provide the initial code to the team."
)

# 2. Code Reviewer Agent (AssistantAgent)
# This agent reviews the code for issues.
code_reviewer = autogen.AssistantAgent(
    name="Code_Reviewer",
    llm_config=llm_config,
    system_message=(
        "You are an expert Python code reviewer. Your task is to meticulously examine the provided code "
        "for bugs, style issues (PEP 8), performance bottlenecks, and potential security vulnerabilities. "
        "Provide detailed, constructive feedback. If you find issues, clearly state them and suggest improvements. "
        "If the code looks good, state 'Code looks good!' and explain why. "
        "Your feedback is crucial for the Debugger/Fixer agent."
    )
)

# 3. Debugger/Fixer Agent (AssistantAgent)
# This agent takes feedback and attempts to fix the code.
debugger_fixer = autogen.AssistantAgent(
    name="Debugger_Fixer",
    llm_config=llm_config,
    system_message=(
        "You are a skilled Python debugger and code fixer. Your role is to receive code and "
        "feedback from the Code Reviewer. Based on the feedback, you must generate a corrected "
        "version of the code that addresses all identified issues. Present the full corrected code. "
        "If no issues are found, acknowledge that."
    )
)

# 4. Verifier Agent (AssistantAgent)
# This agent verifies the corrected code. It reasons about the fix, rather than executing.
verifier = autogen.AssistantAgent(
    name="Verifier",
    llm_config=llm_config,
    system_message=(
        "You are a meticulous code verifier. Your task is to evaluate the corrected code provided "
        "by the Debugger/Fixer agent against the original issues reported by the Code Reviewer. "
        "You will 'simulate' running the code by reasoning about its logic and correctness. "
        "If the fix successfully addresses all issues without introducing new ones, respond with 'PASS'. "
        "Otherwise, respond with 'FAIL' and explain why the fix is insufficient or incorrect. "
        "Your 'PASS' message will terminate the conversation."
    )
)

# --- GroupChat and GroupChatManager Setup ---

# Define the group chat with all agents.
# The `messages` list will store the conversation history.
# `max_round` prevents infinite loops.
# `speaker_selection_method` can be 'auto' or 'round_robin'. 'auto' is generally more flexible.
groupchat = autogen.GroupChat(
    agents=[developer, code_reviewer, debugger_fixer, verifier],
    messages=[],
    max_round=10, # Limit the number of turns to prevent infinite loops
    speaker_selection_method="auto", # AutoGen decides who speaks next based on context
    allow_repeat_speaker=False # Prevent the same agent from speaking twice in a row
)

# Create the GroupChatManager to orchestrate the conversation.
# The manager uses the `llm_config` to decide which agent should speak next.
# `is_termination_msg` is crucial here: the conversation ends when the Verifier says 'PASS'.
manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
    is_termination_msg=lambda x: x.get("content", "").lower().find("pass") != -1 or \
                                 x.get("content", "").lower().find("terminate") != -1
)

# --- Initiate the Conversation ---

print("\n--- Starting Multi-Agent Code Review and Debugging Process ---")

# The Developer agent initiates the chat by providing the problematic code.
# The manager will then take over and orchestrate the conversation among the agents.
chat_result = developer.initiate_chat(
    manager,
    message=f"Here is a Python function I've written. Please review it for any bugs or improvements, and then ensure it's fixed and verified:\n\n```python\n{problematic_code}\n```"
)

print("\n--- Multi-Agent Process Finished ---")
print("Final chat summary:")
print(chat_result.summary)
print("Last message from the group chat:")
print(chat_result.last_message)
